# Example: "Anamak" tokamak
---

Here, we generate a few different equilibria using the "Anamak" example tokamak from the [FGE code paper](https://arxiv.org/abs/2512.06847).

The equilbirium\profile parameters are **completely made up** based on some equilbiria in the above paper - please experiment on your own and change them to more realistic values as you please!

### Create the machine object

The Anamak machine is a useful toy tokamak because the geometry can be varied in a simple and controlled way. In particular, the number and placement of active coils, passive structures, and the vessel can all be adjusted using a small set of geometric parameters.

Following the parameterisation used in the reference above, the $N_{\mathrm{coils}}$ **active coils** are placed on an **ellipse** centred at $(R_0, 0)$ in the poloidal $(R, Z)$ plane. Their coordinates are defined by

$$
R_{\mathrm{coil}}^{(i)} = R_0 + \hat{R}_{\mathrm{coil}} \cos(\theta_i),
$$

$$
Z_{\mathrm{coil}}^{(i)} = \kappa \, \hat{R}_{\mathrm{coil}} \sin(\theta_i),
$$

where
- $R_0$ is the major radius at which the ellipse is centred.
- $\hat{R}_{\mathrm{coil}}$ is the minor radius of the coils. 
- $\bold{\theta}$ is a vector (length $N_{\mathrm{coils}}$) of poloidal angles $\theta \in [0, 2\pi)$. 
- $\kappa$ is the "elongation" of the ellipse.

Similarly, the $N_{\mathrm{passives}}$ **passive structures** are placed on an ellipse at the same centre:

$$
R_{\mathrm{passives}}^{(i)} = R_0 + \hat{R}_{\mathrm{passive}} \cos(\theta_i),
$$

$$
Z_{\mathrm{passives}}^{(i)} = \kappa \, \hat{R}_{\mathrm{passive}} \sin(\theta_i),
$$

but with a potentially different minor radius $ \hat{R}_{\mathrm{passive}}$. 

Finally the $N_{\mathrm{limiter}}$ limiter/wall points are set up similarly:

$$
R_{\mathrm{limiter}}^{(i)} = R_0 + \hat{R}_{\mathrm{limiter}} \cos(\theta_i),
$$

$$
Z_{\mathrm{limiter}}^{(i)} = \kappa \, \hat{R}_{\mathrm{limiter}} \sin(\theta_i).
$$

We should note that given we are only doing static solves below, the passive structures are not really used here.

In the next cell, we can choose the parameters for building the Anamak.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# major radius of the machine
R0 = 1.0

# "elongation" of the machine (same for actives, passives, and limiter here)
kappa = 1.0    

# for each of the coils, passives, and limiter, define:
# -> N = the number of each you desire
# -> R = the minor radius of its ellipse

# actives parameters
N_actives = 8
R_coil = 0.7

# passives parameters
N_passives = 50
R_passive = 0.5

# limiter parameters
N_limiter = 101
R_limiter = 0.45

The following functions are used to build the Anamak.

In [ ]:
# builds an ellipse of (R,Z) points
def build_ellipse(
    R0,        # major radius
    N,         # number of points in [0,2\pi)
    R_minor,   # minor radius
    kappa,     # "elongation"
    ):
    
    # poloidal locations
    theta = np.arange(0, 2*np.pi, 2*np.pi/N)
    
    # locations
    r_locs = R0 + R_minor*np.cos(theta)
    z_locs = R_minor*kappa*np.sin(theta)

    return r_locs, z_locs

# builds active coils using (R,Z) locations of filaments
def build_actives(
    r_locs, 
    z_locs, 
    ):

    # populate active coils dictionary with single filament coils (point-sources)
    active_coils_data = {}
    for i, (r, z) in enumerate(zip(r_locs, z_locs)):
        active_coils_data[i] = {
            "R": [r],
            "Z": [z],
            "dR": 0.05,             # size only used for plotting in the simulations below
            "dZ": 0.05,             # size only used for plotting in the simulations below
            "resistivity": 1.55e-8, # not used below
            "polarity": 1,          # default
            "multiplier": 1,        # single turn coils
        }
        
    return active_coils_data

# builds passive structures using (R,Z) locations of filaments
def build_passives(
    r_locs, 
    z_locs, 
    ):

    # populate passive coils list with single filament passives (point-sources)
    passive_coils_data = []
    for r, z in zip(r_locs, z_locs):
        passive_coils_data.append({
            "R": r,
            "Z": z,
            "dR": 0.02,             # size only used for plotting in the simulations below
            "dZ": 0.02,             # size only used for plotting in the simulations below
            "resistivity": 5.5e-7   # resistivity of material (steel)
        })
        
    return passive_coils_data

# builds limiter geometry using (R,Z) locations
def build_limiter(
    r_locs, 
    z_locs, 
    ):

    # populate limiter list
    limiter_data = []
    for r, z in zip(r_locs, z_locs):
        limiter_data.append({"R": r, "Z": z})
        
    return limiter_data

In [ ]:
# build the active coils data
r_active_locs, z_active_locs = build_ellipse(
    R0=R0, 
    N=N_actives, 
    R_minor=1.4*R_passive,
    kappa=kappa,
)

active_coils_data = build_actives(
    r_locs=r_active_locs, 
    z_locs=z_active_locs
    )

# build the passive structures data
r_passive_locs, z_passive_locs = build_ellipse(
    R0=R0, 
    N=N_passives, 
    R_minor=R_passive,
    kappa=kappa,
)

passive_coils_data = build_passives(
    r_locs=r_passive_locs, 
    z_locs=z_passive_locs
    )

# build the limiter data
r_limiter_locs, z_limiter_locs = build_ellipse(
    R0=R0, 
    N=N_limiter, 
    R_minor=R_limiter,
    kappa=kappa,
)

limiter_data = build_limiter(
    r_locs=r_limiter_locs, 
    z_locs=z_limiter_locs
    )

In [ ]:
# build the Anamak
from freegsnke import build_machine
tokamak = build_machine.tokamak(
    active_coils_data=active_coils_data,
    passive_coils_data=passive_coils_data,
    limiter_data=limiter_data,
    wall_data=limiter_data,
)

# plot the machine
fig1, ax1 = plt.subplots(1, 1, figsize=(7, 15), dpi=80)
ax1.grid(zorder=0, alpha=0.75)
ax1.set_aspect('equal')
tokamak.plot(axis=ax1,show=False)                                                         
ax1.fill(tokamak.wall.R, tokamak.wall.Z, color='k', linewidth=1.2, facecolor='w', zorder=0) 
ax1.set_xlabel("Major radius [m]")
ax1.set_ylabel("Height [m]")

### Instantiate an equilibrium

In [ ]:
from freegsnke import equilibrium_update

eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,      # provide tokamak object
    Rmin=0.1, Rmax=1.9,   # radial range
    Zmin=-0.8, Zmax=0.8,  # vertical range
    nx=65,                # number of grid points in the radial direction (needs to be of the form (2**n + 1) with n being an integer)
    ny=129,               # number of grid points in the vertical direction (needs to be of the form (2**n + 1) with n being an integer)
)

### Instantiate a profile object

We use the Lao profile object here, providing constant pressure and toroidal field profiles for simplicity. 

In [ ]:
from freegsnke.jtor_update import Lao85
profiles = Lao85(
    eq=eq,             # eq object
    Ip=2e5,            # plasma current
    fvac=0.5*R0,       # toroidal field * major radius
    alpha=[58213.6],   # linear profile
    beta=[0.582136]    # linear profile
)

### Load the static nonlinear solver

In [ ]:
from freegsnke import GSstaticsolver
GSStaticSolver = GSstaticsolver.NKGSsolver(eq)    

### Inverse solver constraints

In this first example, we'll set up some simple constraints to generate an up/down symmetric double null plasma (with relatively high triangularity). 

In [ ]:
# null points constraints
Rx = 0.8      # X-point radius
Zx = 0.35     # X-point height
null_points = [[Rx, Rx], [Zx, -Zx]]

# isoflux constraints (including null points here)
Rout = 1.22    # outboard midplane radius
isoflux_set = np.array([
    [
    [Rx, Rx, Rout], 
    [Zx, -Zx, 0.0]
    ]]
                       )
           
# instantiate the freegsnke constrain object
from freegsnke.inverse import Inverse_optimizer
constrain = Inverse_optimizer(
    null_points=null_points,
    isoflux_set=isoflux_set,
    weight_isoflux=1.0,
    weight_nulls=0.8,    # slightly lower weight on null points to get correct shape
    )

### The inverse solve

Solve using the inverse solver with forced up/down symmetry. 

In [ ]:
GSStaticSolver.inverse_solve(eq=eq, 
                     profiles=profiles, 
                     constrain=constrain, 
                     target_relative_tolerance=1e-5,
                     target_relative_psit_update=1e-3,
                     verbose=True,
                     l2_reg=np.array([1e-12]*N_actives),
                     force_up_down_symmetric=True,
                     )

In [ ]:
# refine with a forward solve (now the currents are known)
GSStaticSolver.solve(eq=eq, 
                     profiles=profiles, 
                     constrain=None, 
                     target_relative_tolerance=1e-9,
                     verbose=False, # print output
                     )

Using only a few constraints, we can see that the inverse solver has found the desired plasma. It is almost perfectly up/down symmetric and has high triangularity. 

In [ ]:
print(f"Triangularity = {eq.triangularity()}")
print(f"Avg. Z (Jtor) position = {eq.Zcurrent()} [m]")

In [ ]:
# plot the equilibrium
fig1, ax1 = plt.subplots(1, 1, figsize=(7, 15), dpi=80)
ax1.grid(zorder=0, alpha=0.75)
ax1.set_aspect('equal')
eq.tokamak.plot(axis=ax1,show=False)                                                          # plots the active coils and passive structures
ax1.fill(tokamak.wall.R, tokamak.wall.Z, color='k', linewidth=1.2, facecolor='w', zorder=0)   # plots the limiter
eq.plot(axis=ax1,show=False)                                                                  # plots the equilibrium
constrain.plot(axis=ax1, show=False)                                                          # plots the contraints

Next, we'll use the same machine description to make a negtive triangularity plasma (again up/down symmetric). 

In [ ]:
# null points constraints
Rx = 1.1      # X-point radius
Zx = 0.35      # X-point height
null_points = [[Rx, Rx], [Zx, -Zx]]

# isoflux constraints (including null points here)
Rin = 0.8    # inboard midplane radius
isoflux_set = np.array([
    [
    [Rx, Rx, Rin], 
    [Zx, -Zx, 0.0]
    ]]
                       )
           
# instantiate the freegsnke constrain object
from freegsnke.inverse import Inverse_optimizer
constrain = Inverse_optimizer(
    null_points=null_points,
    isoflux_set=isoflux_set,
    weight_isoflux=1.0,
    weight_nulls=0.8,    # slightly lower weight on null points to get correct shape
    )

In [ ]:
# solve
GSStaticSolver.inverse_solve(eq=eq, 
                     profiles=profiles, 
                     constrain=constrain, 
                     target_relative_tolerance=1e-5,
                     target_relative_psit_update=1e-3,
                     verbose=True,
                     l2_reg=np.array([1e-14]*N_actives),
                     force_up_down_symmetric=True,
                     )

In [ ]:
# refine with a forward solve (now the currents are known)
GSStaticSolver.solve(eq=eq, 
                     profiles=profiles, 
                     constrain=None, 
                     target_relative_tolerance=1e-9,
                     verbose=False, # print output
                     )

In [ ]:
print(f"Triangularity = {eq.triangularity()}")
print(f"Avg. Z (Jtor) position = {eq.Zcurrent()} [m]")

In [ ]:
# plot the equilibrium
fig1, ax1 = plt.subplots(1, 1, figsize=(7, 15), dpi=80)
ax1.grid(zorder=0, alpha=0.75)
ax1.set_aspect('equal')
eq.tokamak.plot(axis=ax1,show=False)                                                          # plots the active coils and passive structures
ax1.fill(tokamak.wall.R, tokamak.wall.Z, color='k', linewidth=1.2, facecolor='w', zorder=0)   # plots the limiter
eq.plot(axis=ax1,show=False)                                                                  # plots the equilibrium
constrain.plot(axis=ax1, show=False)                                                          # plots the contraints

Now lets look at changing the shape of the Anamak.

We'll now increase the number of active coils and increase the "elongation" of the device. 

In [ ]:
# "elongation" of the machine (same for actives, passives, and limiter here)
kappa = 1.8

# for each of the coils, passives, and limiter, define:
# -> N = the number of each you desire
# -> R = the minor radius of its ellipse

# actives parameters
N_actives = 24
R_coil = 0.7

In [ ]:
# build the active coils data
r_active_locs, z_active_locs = build_ellipse(
    R0=R0, 
    N=N_actives, 
    R_minor=1.4*R_passive,
    kappa=kappa,
)

active_coils_data = build_actives(
    r_locs=r_active_locs, 
    z_locs=z_active_locs
    )

# build the passive structures data
r_passive_locs, z_passive_locs = build_ellipse(
    R0=R0, 
    N=N_passives, 
    R_minor=R_passive,
    kappa=kappa,
)

passive_coils_data = build_passives(
    r_locs=r_passive_locs, 
    z_locs=z_passive_locs
    )

# build the limiter data
r_limiter_locs, z_limiter_locs = build_ellipse(
    R0=R0, 
    N=N_limiter, 
    R_minor=R_limiter,
    kappa=kappa,
)

limiter_data = build_limiter(
    r_locs=r_limiter_locs, 
    z_locs=z_limiter_locs
    )

In [ ]:
# build the Anamak
from freegsnke import build_machine
tokamak = build_machine.tokamak(
    active_coils_data=active_coils_data,
    passive_coils_data=passive_coils_data,
    limiter_data=limiter_data,
    wall_data=limiter_data,
)

# plot the machine
fig1, ax1 = plt.subplots(1, 1, figsize=(7, 15), dpi=80)
ax1.grid(zorder=0, alpha=0.75)
ax1.set_aspect('equal')
tokamak.plot(axis=ax1,show=False)                                                         
ax1.fill(tokamak.wall.R, tokamak.wall.Z, color='k', linewidth=1.2, facecolor='w', zorder=0) 
ax1.set_xlabel("Major radius [m]")
ax1.set_ylabel("Height [m]")

Re-initialise the equilibrium and profiles objects: here we've increased the vertical size of the computational grid and increased the plasma current. 

In [ ]:
from freegsnke import equilibrium_update

eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,      # provide tokamak object
    Rmin=0.1, Rmax=1.9,   # radial range
    Zmin=-1.5, Zmax=1.5,  # vertical range
    nx=65,                # number of grid points in the radial direction (needs to be of the form (2**n + 1) with n being an integer)
    ny=129,               # number of grid points in the vertical direction (needs to be of the form (2**n + 1) with n being an integer)
)

from freegsnke.jtor_update import Lao85
profiles = Lao85(
    eq=eq,             # eq object
    Ip=5e5,            # plasma current
    fvac=0.5*R0,       # toroidal field * major radius
    alpha=[58213.6],   # linear profile
    beta=[0.582136]    # linear profile
)

from freegsnke import GSstaticsolver
GSStaticSolver = GSstaticsolver.NKGSsolver(eq)

Let's try to create a simple up/down symmetric double null plasma in this new machine. 

In [ ]:
# null points constraints
Rx = 0.8      # X-point radius
Zx = 0.65     # X-point height
null_points = [[Rx, Rx], [Zx, -Zx]]

# isoflux constraints (including null points here)
Rout = 1.3    # outboard midplane radius
isoflux_set = np.array([
    [
    [Rx, Rx, Rout], 
    [Zx, -Zx, 0.0]
    ]]
                       )
           
# instantiate the freegsnke constrain object
from freegsnke.inverse import Inverse_optimizer
constrain = Inverse_optimizer(
    null_points=null_points,
    isoflux_set=isoflux_set,
    weight_isoflux=1.0,
    weight_nulls=0.7,    # slightly lower weight on null points to get correct shape
    )

In [ ]:
# solve
GSStaticSolver.inverse_solve(eq=eq, 
                     profiles=profiles, 
                     constrain=constrain, 
                     target_relative_tolerance=1e-5,
                     target_relative_psit_update=1e-3,
                     verbose=True,
                     l2_reg=np.array([1e-12]*N_actives),
                     force_up_down_symmetric=True,
                     )

In [ ]:
# refine with a forward solve (now the currents are known)
GSStaticSolver.solve(eq=eq, 
                     profiles=profiles, 
                     constrain=None, 
                     target_relative_tolerance=1e-9,
                     verbose=False, # print output
                     )

In [ ]:
# plot the equilibrium
fig1, ax1 = plt.subplots(1, 1, figsize=(7, 15), dpi=80)
ax1.grid(zorder=0, alpha=0.75)
ax1.set_aspect('equal')
eq.tokamak.plot(axis=ax1,show=False)                                                          # plots the active coils and passive structures
ax1.fill(tokamak.wall.R, tokamak.wall.Z, color='k', linewidth=1.2, facecolor='w', zorder=0)   # plots the limiter
eq.plot(axis=ax1,show=False)                                                                  # plots the equilibrium
constrain.plot(axis=ax1, show=False)                                                          # plots the contraints